# Phase 0 Dataset Sanity Check

Use this notebook to inspect a CSV or Parquet export of the facial landmark dataset.

What it checks:
- sample shape and column layout
- label column presence and counts
- grouping columns such as `video_filename` and `frame_num`
- a quick normalization drift probe on the first row

If your dataset lives in Google Drive, mount the drive first and update `DATASET_PATH` below.

In [ ]:
!pip -q install numpy pandas pyarrow


In [ ]:
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


DATASET_EXTENSIONS = {".csv", ".parquet", ".pq"}
LABEL_CANDIDATES = ("label", "emotion", "emotion_label", "class")
GROUP_CANDIDATES = ("video_filename", "frame_num")
DEFAULT_ANCHOR_INDICES = (1, 6, 168)
DEFAULT_SCALE_INDICES = (33, 263)


def _read_sample(dataset_path: Path, rows: int) -> pd.DataFrame:
    suffix = dataset_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(dataset_path, nrows=rows)
    if suffix in {".parquet", ".pq"}:
        frame = pd.read_parquet(dataset_path)
        return frame.head(rows)
    raise ValueError(f"Unsupported dataset format: {dataset_path.suffix}")


def _pick_column(columns: Iterable[str], candidates: Iterable[str]) -> str | None:
    available = set(columns)
    for candidate in candidates:
        if candidate in available:
            return candidate
    return None


def _as_landmark_array(landmarks) -> np.ndarray:
    array = np.asarray(landmarks, dtype=np.float32)
    if array.ndim != 2 or array.shape[1] != 3:
        raise ValueError(f"Expected landmark array shaped (n, 3), got {array.shape}")
    return array


def normalize_landmarks(landmarks, anchor_indices=DEFAULT_ANCHOR_INDICES, scale_indices=DEFAULT_SCALE_INDICES):
    array = _as_landmark_array(landmarks)
    anchor_points = array[list(anchor_indices)]
    anchor = anchor_points.mean(axis=0)

    scale_points = array[list(scale_indices)]
    if len(scale_points) < 2:
        raise ValueError("Scale estimation needs at least two landmark indices")

    scale = float(np.linalg.norm(scale_points[0] - scale_points[1]))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0

    normalized = (array - anchor) / scale
    return normalized, {"anchor": anchor, "scale": scale}


def landmark_block_to_array(frame_row, prefix: str = "") -> np.ndarray:
    if hasattr(frame_row, "to_dict"):
        values = frame_row.to_dict()
    elif isinstance(frame_row, dict):
        values = frame_row
    else:
        raise TypeError("frame_row must be a mapping-like object")

    x_values = []
    y_values = []
    z_values = []

    index = 0
    while True:
        x_key = f"{prefix}x_{index}"
        y_key = f"{prefix}y_{index}"
        z_key = f"{prefix}z_{index}"
        if x_key not in values or y_key not in values or z_key not in values:
            break
        x_values.append(values[x_key])
        y_values.append(values[y_key])
        z_values.append(values[z_key])
        index += 1

    if not x_values:
        raise ValueError("No landmark columns found with the expected x_N/y_N/z_N pattern")

    return np.column_stack([x_values, y_values, z_values]).astype(np.float32)


def summarize_normalization_drift(landmarks, anchor_indices=DEFAULT_ANCHOR_INDICES, scale_indices=DEFAULT_SCALE_INDICES):
    normalized, stats = normalize_landmarks(landmarks, anchor_indices, scale_indices)
    centroid = normalized.mean(axis=0)
    spread = normalized.std(axis=0)
    return {
        "scale": stats["scale"],
        "centroid_norm": float(np.linalg.norm(centroid)),
        "spread_mean": float(spread.mean()),
    }


def inspect_dataset(dataset_path: Path, rows: int) -> int:
    sample = _read_sample(dataset_path, rows)
    print(f"Loaded sample shape: {sample.shape}")
    print("Columns:")
    print(", ".join(sample.columns[:20]))

    label_column = _pick_column(sample.columns, LABEL_CANDIDATES)
    if label_column:
        label_counts = sample[label_column].value_counts(dropna=False).to_dict()
        print(f"Label column: {label_column}")
        print(f"Label counts: {label_counts}")
    else:
        print("Label column: not found")

    group_hits = [column for column in GROUP_CANDIDATES if column in sample.columns]
    print(f"Grouping columns present: {group_hits or 'none'}")

    landmark_row = sample.iloc[0]
    landmark_array = landmark_block_to_array(landmark_row)
    drift = summarize_normalization_drift(landmark_array)
    print("Normalization probe:")
    print(drift)

    expected_feature_count = landmark_array.shape[0] * 3
    observed_feature_columns = [column for column in sample.columns if column.startswith(("x_", "y_", "z_"))]
    print(f"Landmark count: {landmark_array.shape[0]}")
    print(f"Observed landmark feature columns: {len(observed_feature_columns)}")
    print(f"Expected landmark feature columns from row probe: {expected_feature_count}")
    return 0


In [ ]:
DATASET_PATH = Path("/content/your_dataset.parquet")
SAMPLE_ROWS = 5

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Set DATASET_PATH to a real CSV or Parquet file. Current value does not exist: {DATASET_PATH}"
    )

inspect_dataset(DATASET_PATH, SAMPLE_ROWS)
